# Pascal VOC 2012 Multi-Label Image Classification

End-to-end multi-label classification on Pascal VOC 2012 using modern PyTorch.

**Architecture choices:** EfficientNet-B3 backbone (ImageNet-pretrained) with a
multi-label classification head, trained with mixed precision (torch.amp),
AdamW + OneCycleLR schedule, and label-smoothed BCE loss.

**Dataset:** Pascal VOC 2012 — 20 object categories, one image can have multiple labels.

**Metric:** Mean Average Precision (mAP).

In [7]:
# Install dependencies (run once on Colab)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "torchvision", "scikit-learn", "matplotlib",
                "numpy", "Pillow", "tqdm"], check=True)
print("Dependencies ready.")

Dependencies ready.


In [8]:
import os, json, random, math, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms.v2 as T
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import average_precision_score
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.10.0+cu128 | torchvision 0.25.0+cu128 | device: cuda
GPU: Tesla T4


## Configuration

In [9]:
# ── Paths ──────────────────────────────────────────────────────────────────
ON_COLAB = False
try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    pass

if ON_COLAB:
    drive.mount("/content/gdrive")
    _DRIVE_DATA = Path("/content/gdrive/My Drive/pascal12/data")
    MODEL_DIR   = Path("/content/gdrive/My Drive/pascal12/models")
    # Use Drive data if it already exists, otherwise download locally
    DATA_ROOT = _DRIVE_DATA if _DRIVE_DATA.exists() else Path("/content/voc/VOCdevkit/VOC2012")
else:
    DATA_ROOT = Path("./data/VOCdevkit/VOC2012")
    MODEL_DIR = Path("./models")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyper-parameters ────────────────────────────────────────────────────────
IMG_SIZE     = 300
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LR_MAX       = 1e-3
WEIGHT_DECAY = 1e-4
NUM_CLASSES  = 20
NUM_WORKERS  = 2

VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MODEL_DIR: {MODEL_DIR}")
print(f"Classes ({NUM_CLASSES}):", VOC_CLASSES)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
DATA_ROOT: /content/voc/VOCdevkit/VOC2012
MODEL_DIR: /content/gdrive/My Drive/pascal12/models
Classes (20): ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']


## Data Download

Downloads Pascal VOC 2012 via torchvision if `DATA_ROOT` does not already exist.

In [10]:
import torchvision.datasets as _voc_ds

if not DATA_ROOT.exists():
    print(f"DATA_ROOT not found at {DATA_ROOT}. Downloading Pascal VOC 2012 ...")
    _dl_root = DATA_ROOT.parent.parent  # e.g. /content/voc
    _dl_root.mkdir(parents=True, exist_ok=True)
    # torchvision handles download + extraction; we only need the side-effect
    _voc_ds.VOCDetection(root=str(_dl_root), year="2012", image_set="train",
                          download=True)
    DATA_ROOT = _dl_root / "VOCdevkit" / "VOC2012"
    print(f"Download complete. DATA_ROOT set to: {DATA_ROOT}")
else:
    print(f"Data found at: {DATA_ROOT}")

assert DATA_ROOT.exists(), f"DATA_ROOT still missing: {DATA_ROOT}"
assert (DATA_ROOT / "ImageSets" / "Main" / "train.txt").exists(), \
    "VOC split files not found — check DATA_ROOT structure."

DATA_ROOT not found at /content/voc/VOCdevkit/VOC2012. Downloading Pascal VOC 2012 ...


100%|██████████| 2.00G/2.00G [01:21<00:00, 24.6MB/s] 


Download complete. DATA_ROOT set to: /content/voc/VOCdevkit/VOC2012


## Dataset

In [11]:
class PascalVOCDataset(Dataset):
    """Multi-label Pascal VOC 2012 dataset."""

    def __init__(self, root: Path, split: str = "train", transform=None):
        self.root = root
        self.transform = transform
        self.img_dir = root / "JPEGImages"
        self.ann_dir = root / "Annotations"

        split_file = root / "ImageSets" / "Main" / f"{split}.txt"
        with open(split_file) as f:
            self.ids = [line.strip() for line in f if line.strip()]

    def __len__(self):
        return len(self.ids)

    def _encode_labels(self, xml_path: Path) -> torch.Tensor:
        label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        tree = ET.parse(xml_path)
        for obj in tree.getroot().iter("object"):
            name = obj.find("name").text
            if name in VOC_CLASSES:
                label[VOC_CLASSES.index(name)] = 1.0
        return label

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = Image.open(self.img_dir / f"{img_id}.jpg").convert("RGB")
        label = self._encode_labels(self.ann_dir / f"{img_id}.xml")
        if self.transform:
            img = self.transform(img)
        return img, label, img_id

print("PascalVOCDataset defined.")

PascalVOCDataset defined.


## Transforms (torchvision.transforms.v2)

In [12]:
# Training: strong augmentation via RandAugment + geometric transforms
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandAugment(num_ops=2, magnitude=9),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Validation / test: deterministic centre-crop pipeline
val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Transforms ready.")

Transforms ready.


In [13]:
train_dataset = PascalVOCDataset(DATA_ROOT, split="train", transform=train_transform)
val_dataset   = PascalVOCDataset(DATA_ROOT, split="val",   transform=val_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"Train: {len(train_dataset):,} images | Val: {len(val_dataset):,} images")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train: 5,717 images | Val: 5,823 images
Train batches: 178 | Val batches: 182


## Model — EfficientNet-B3 Backbone

In [14]:
def build_model(backbone: str = "efficientnet_b3", num_classes: int = NUM_CLASSES) -> nn.Module:
    """Return a pretrained backbone with a multi-label classification head."""
    if backbone == "efficientnet_b3":
        weights = EfficientNet_B3_Weights.DEFAULT
        model = efficientnet_b3(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes)
        )
    elif backbone == "resnet50":
        weights = ResNet50_Weights.DEFAULT
        model = resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes)
        )
    else:
        raise ValueError(f"Unknown backbone: {backbone}")
    return model


model = build_model("efficientnet_b3")
model = model.to(DEVICE)

# Optional: compile for extra speed on PyTorch 2.0+
if hasattr(torch, "compile"):
    model = torch.compile(model)
    print("torch.compile enabled.")

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters — total: {total_params:,} | trainable: {train_params:,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 118MB/s] 


torch.compile enabled.
Parameters — total: 10,726,972 | trainable: 10,726,972


## Loss, Optimizer, Scheduler

In [15]:
def label_smoothed_bce(pred: torch.Tensor, target: torch.Tensor,
                        smoothing: float = 0.05) -> torch.Tensor:
    """BCE with label smoothing for multi-label classification."""
    target_smooth = target * (1.0 - smoothing) + smoothing * 0.5
    return nn.functional.binary_cross_entropy_with_logits(pred, target_smooth)


# Differential learning rates: lower LR for pretrained backbone, higher for head
def get_param_groups(model: nn.Module, backbone_name: str = "features"):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if backbone_name in name:
            backbone_params.append(p)
        else:
            head_params.append(p)
    return [
        {"params": backbone_params, "lr": LR_MAX / 10},
        {"params": head_params,     "lr": LR_MAX},
    ]


optimizer = optim.AdamW(
    get_param_groups(model),
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR_MAX / 10, LR_MAX],
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1,
    anneal_strategy="cos"
)

scaler = torch.amp.GradScaler(device=DEVICE.type)
print("Optimizer: AdamW | Scheduler: OneCycleLR | Precision: mixed (AMP)")

Optimizer: AdamW | Scheduler: OneCycleLR | Precision: mixed (AMP)


## Training

In [16]:
def compute_map(all_labels: np.ndarray, all_scores: np.ndarray) -> float:
    """Mean Average Precision across classes that have at least one positive sample."""
    aps = []
    for c in range(all_labels.shape[1]):
        if all_labels[:, c].sum() > 0:
            aps.append(average_precision_score(all_labels[:, c], all_scores[:, c]))
    return float(np.mean(aps)) if aps else 0.0


def train_one_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0.0
    for imgs, labels, _ in tqdm(loader, desc="  train", leave=False):
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type):
            logits = model(imgs)
            loss = label_smoothed_bce(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    all_labels, all_scores = [], []
    for imgs, labels, _ in tqdm(loader, desc="  eval ", leave=False):
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type):
            logits = model(imgs)
            loss = label_smoothed_bce(logits, labels)
        total_loss += loss.item()
        all_labels.append(labels.cpu().numpy())
        all_scores.append(torch.sigmoid(logits).cpu().numpy())
    all_labels = np.concatenate(all_labels)
    all_scores = np.concatenate(all_scores)
    return total_loss / len(loader), compute_map(all_labels, all_scores), all_labels, all_scores


print("Training functions defined.")

Training functions defined.


In [1]:
history = {"train_loss": [], "val_loss": [], "val_map": []}
best_map = 0.0
best_model_path = MODEL_DIR / "efficientnet_b3_best.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, DEVICE)
    val_loss, val_map, _, _ = evaluate(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_map"].append(val_map)

    print(f"  loss: train={train_loss:.4f}  val={val_loss:.4f}  mAP={val_map:.4f}")

    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✓ New best mAP={best_map:.4f} — model saved.")

print(f"Training complete. Best mAP: {best_map:.4f}")

NameError: name 'MODEL_DIR' is not defined

## Training Curves

In [ ]:
def plot_history(history: dict):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history["train_loss"], label="Train Loss", lw=2)
    axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   lw=2, linestyle="--")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["val_map"], label="Val mAP", lw=2, color="green")
    axes[1].set_title("Validation mAP"); axes[1].set_xlabel("Epoch")
    axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(MODEL_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_history(history)

## Per-class Average Precision

In [ ]:
# Load best checkpoint and evaluate
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
_, _, all_labels, all_scores = evaluate(model, val_loader, DEVICE)

per_class_ap = [
    average_precision_score(all_labels[:, c], all_scores[:, c])
    if all_labels[:, c].sum() > 0 else 0.0
    for c in range(NUM_CLASSES)
]
mean_ap = np.mean(per_class_ap)

fig, ax = plt.subplots(figsize=(14, 5))
colours = ["steelblue" if ap >= mean_ap else "salmon" for ap in per_class_ap]
bars = ax.bar(VOC_CLASSES, per_class_ap, color=colours)
ax.axhline(mean_ap, color="black", linestyle="--", lw=1.5, label=f"mAP = {mean_ap:.3f}")
ax.set_title("Per-class Average Precision (best checkpoint)")
ax.set_ylim(0, 1); ax.set_ylabel("AP")
ax.set_xticks(range(len(VOC_CLASSES)))
ax.set_xticklabels(VOC_CLASSES, rotation=45, ha="right")
above = mpatches.Patch(color="steelblue", label=">= mAP")
below = mpatches.Patch(color="salmon",    label="< mAP")
ax.legend(handles=[above, below, ax.axhline(mean_ap, color="black", lw=0)], loc="upper right")
plt.tight_layout()
plt.savefig(MODEL_DIR / "per_class_ap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"mAP: {mean_ap:.4f}")

## Visualise Predictions

In [ ]:
def denormalize(tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = tensor.cpu() * std + mean
    return img.permute(1, 2, 0).numpy().clip(0, 1)


def visualize_predictions(
    model: nn.Module,
    dataset: Dataset,
    device: torch.device,
    n_images: int = 8,
    threshold: float = 0.5,
):
    """Show a grid of images with ground-truth and predicted labels."""
    model.eval()
    indices = random.sample(range(len(dataset)), n_images)
    cols = 4
    rows = math.ceil(n_images / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    with torch.no_grad():
        for ax, idx in zip(axes, indices):
            img_t, label, img_id = dataset[idx]
            logits = model(img_t.unsqueeze(0).to(device))
            probs  = torch.sigmoid(logits).squeeze().cpu().numpy()

            gt_labels   = [VOC_CLASSES[i] for i, v in enumerate(label) if v > 0.5]
            pred_labels = [VOC_CLASSES[i] for i, p in enumerate(probs) if p >= threshold]

            ax.imshow(denormalize(img_t))
            ax.set_title(f"{os.path.basename(str(img_id))}"
                f"GT:   {', '.join(gt_labels) or 'none'}"
                f"Pred: {', '.join(pred_labels) or 'none'}",
                fontsize=8
            )
            ax.axis("off")

    for ax in axes[n_images:]:
        ax.axis("off")

    plt.suptitle("Ground Truth vs Predictions", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "predictions_grid.png", dpi=150, bbox_inches="tight")
    plt.show()


# Use val dataset with val_transform (no augmentation) for clean visualisation
visualize_predictions(model, val_dataset, DEVICE, n_images=8)

## Co-occurrence Heatmap (Predicted Positive Labels)

In [ ]:
def cooccurrence_heatmap(all_scores: np.ndarray, threshold: float = 0.5):
    """Show how often two classes are predicted positive together."""
    preds = (all_scores >= threshold).astype(float)
    cooc  = preds.T @ preds  # shape (20, 20)
    # Normalise by diagonal (self-count)
    with np.errstate(divide="ignore", invalid="ignore"):
        cooc_norm = cooc / np.diag(cooc)[:, None]
        cooc_norm = np.nan_to_num(cooc_norm)

    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cooc_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(VOC_CLASSES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(VOC_CLASSES, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title("Predicted Label Co-occurrence (row-normalised)")
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "cooccurrence.png", dpi=150, bbox_inches="tight")
    plt.show()

cooccurrence_heatmap(all_scores)

## Optional: ResNet-50 Baseline Comparison

Run this cell to train a ResNet-50 baseline and compare mAP side-by-side.

In [ ]:
COMPARE = False  # Set to True to run the comparison

if COMPARE:
    resnet_model = build_model("resnet50").to(DEVICE)
    resnet_optimizer = optim.AdamW(
        get_param_groups(resnet_model, backbone_name="layer"),
        weight_decay=WEIGHT_DECAY
    )
    resnet_scheduler = optim.lr_scheduler.OneCycleLR(
        resnet_optimizer,
        max_lr=[LR_MAX / 10, LR_MAX],
        steps_per_epoch=len(train_loader),
        epochs=NUM_EPOCHS,
        pct_start=0.1,
    )
    resnet_scaler = torch.amp.GradScaler(device=DEVICE.type)
    resnet_history = {"train_loss": [], "val_loss": [], "val_map": []}

    for epoch in range(1, NUM_EPOCHS + 1):
        tl = train_one_epoch(resnet_model, train_loader, resnet_optimizer,
                             resnet_scheduler, resnet_scaler, DEVICE)
        vl, vm, _, _ = evaluate(resnet_model, val_loader, DEVICE)
        resnet_history["train_loss"].append(tl)
        resnet_history["val_loss"].append(vl)
        resnet_history["val_map"].append(vm)
        print(f"ResNet-50 epoch {epoch} | loss={vl:.4f} | mAP={vm:.4f}")

    # Side-by-side mAP comparison
    epochs = range(1, NUM_EPOCHS + 1)
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["val_map"],        label="EfficientNet-B3", lw=2)
    plt.plot(epochs, resnet_history["val_map"], label="ResNet-50",       lw=2, linestyle="--")
    plt.title("Validation mAP: EfficientNet-B3 vs ResNet-50")
    plt.xlabel("Epoch"); plt.ylabel("mAP")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("Comparison skipped (set COMPARE=True to enable).")